<a href="https://colab.research.google.com/github/mohammedAlkhuzaie/Suha-Ali-Salman/blob/main/Test_car_management_system_Suha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Test Car Management System — Self-Contained Version
======================================================

This file contains the full Document Editor implementation, the full Car
Configuration implementation, the CarManagementSystem facade that combines
them, AND the tests for all of it -- in one file, so it can be pasted into
a single Colab/Jupyter cell and run with no cross-file imports.

In a real project, keep document_editor.py, car_configuration.py,
car_management_system.py, and tests/test_car_management_system.py as
separate files and just make sure they're all present in the same working
directory. That separation only breaks in notebooks, where each cell/file
you paste in is standalone unless you explicitly upload every file it
depends on.

To run in Colab:
    !pip install pytest --quiet
    (paste this whole file into a cell, run it)
"""

from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Set

import pytest


# ---------------------------------------------------------------------------
# Part 1: Document Editor (Factory Method + registry pattern)
# ---------------------------------------------------------------------------

class Document(ABC):
    """Abstract contract that every document format must implement."""

    def __init__(self, content: str = "") -> None:
        self.content = content

    @abstractmethod
    def save(self, path: str) -> str:
        raise NotImplementedError

    @abstractmethod
    def render(self) -> str:
        raise NotImplementedError

    @property
    @abstractmethod
    def extension(self) -> str:
        raise NotImplementedError


class PDFDocument(Document):
    def save(self, path: str) -> str:
        return f"{path}{self.extension}"

    def render(self) -> str:
        return f"[PDF] {self.content}"

    @property
    def extension(self) -> str:
        return ".pdf"


class WordDocument(Document):
    def save(self, path: str) -> str:
        return f"{path}{self.extension}"

    def render(self) -> str:
        return f"[WORD] {self.content}"

    @property
    def extension(self) -> str:
        return ".docx"


class HTMLDocument(Document):
    def save(self, path: str) -> str:
        return f"{path}{self.extension}"

    def render(self) -> str:
        return f"<html><body>{self.content}</body></html>"

    @property
    def extension(self) -> str:
        return ".html"


class UnsupportedFormatError(ValueError):
    """Raised when the editor is asked for a format that has not been registered."""


class DocumentFactory:
    """Registry-based factory. New formats register themselves; the factory
    and the editor never need an if/elif chain or any modification."""

    _registry: Dict[str, Callable[[str], Document]] = {}

    @classmethod
    def register(cls, format_name: str, creator: Callable[[str], Document]) -> None:
        cls._registry[format_name.lower()] = creator

    @classmethod
    def create(cls, format_name: str, content: str = "") -> Document:
        key = format_name.lower()
        if key not in cls._registry:
            raise UnsupportedFormatError(
                f"No document type registered for format '{format_name}'. "
                f"Available: {sorted(cls._registry)}"
            )
        return cls._registry[key](content)

    @classmethod
    def available_formats(cls) -> List[str]:
        return sorted(cls._registry)


DocumentFactory.register("pdf", PDFDocument)
DocumentFactory.register("word", WordDocument)
DocumentFactory.register("html", HTMLDocument)


class DocumentEditor:
    """Core editor logic. Depends only on the Document abstraction and the
    factory -- never on a concrete format class."""

    def __init__(self) -> None:
        self._document: Optional[Document] = None

    def new_document(self, format_name: str, content: str = "") -> Document:
        self._document = DocumentFactory.create(format_name, content)
        return self._document

    def edit(self, content: str) -> None:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        self._document.content = content

    def display(self) -> str:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        return self._document.render()

    def save(self, path: str) -> str:
        if self._document is None:
            raise RuntimeError("No document open. Call new_document() first.")
        return self._document.save(path)

    @property
    def current_document(self) -> Optional[Document]:
        return self._document


# ---------------------------------------------------------------------------
# Part 2: Car Configuration (Builder pattern)
# ---------------------------------------------------------------------------

class InvalidConfigurationError(ValueError):
    """Raised when build() is called on a configuration that is incomplete
    or that selects an option the model does not support."""


@dataclass(frozen=True)
class CarModelSpec:
    name: str
    allowed_engines: Set[str]
    allowed_transmissions: Set[str]
    allowed_interior_features: Set[str]
    allowed_exterior_options: Set[str]
    allowed_safety_features: Set[str]
    required_engine: bool = True
    required_transmission: bool = True
    required_safety_features: Set[str] = field(default_factory=set)


@dataclass(frozen=True)
class Car:
    model: str
    engine: Optional[str]
    transmission: Optional[str]
    interior_features: List[str]
    exterior_options: Dict[str, str]
    safety_features: List[str]

    def summary(self) -> str:
        lines = [f"Car order: {self.model}"]
        lines.append(f"  Engine: {self.engine or '—'}")
        lines.append(f"  Transmission: {self.transmission or '—'}")
        lines.append(f"  Interior: {', '.join(self.interior_features) or 'none'}")
        lines.append(
            f"  Exterior: {', '.join(f'{k}={v}' for k, v in self.exterior_options.items()) or 'none'}"
        )
        lines.append(f"  Safety: {', '.join(self.safety_features) or 'none'}")
        return "\n".join(lines)


class CarBuilder:
    """Step-by-step, fluent, flexible builder. Any step may be skipped;
    build() validates the result against the model's spec before returning
    a Car."""

    def __init__(self, spec: CarModelSpec) -> None:
        self._spec = spec
        self._engine: Optional[str] = None
        self._transmission: Optional[str] = None
        self._interior_features: List[str] = []
        self._exterior_options: Dict[str, str] = {}
        self._safety_features: List[str] = []

    def set_engine(self, engine: str) -> "CarBuilder":
        if engine not in self._spec.allowed_engines:
            raise InvalidConfigurationError(
                f"Engine '{engine}' is not offered on {self._spec.name}. "
                f"Choose from {sorted(self._spec.allowed_engines)}."
            )
        self._engine = engine
        return self

    def set_transmission(self, transmission: str) -> "CarBuilder":
        if transmission not in self._spec.allowed_transmissions:
            raise InvalidConfigurationError(
                f"Transmission '{transmission}' is not offered on {self._spec.name}. "
                f"Choose from {sorted(self._spec.allowed_transmissions)}."
            )
        self._transmission = transmission
        return self

    def add_interior_feature(self, feature: str) -> "CarBuilder":
        if feature not in self._spec.allowed_interior_features:
            raise InvalidConfigurationError(
                f"Interior feature '{feature}' is not offered on {self._spec.name}."
            )
        if feature not in self._interior_features:
            self._interior_features.append(feature)
        return self

    def set_exterior_option(self, key: str, value: str) -> "CarBuilder":
        if key not in self._spec.allowed_exterior_options:
            raise InvalidConfigurationError(
                f"Exterior option '{key}' is not offered on {self._spec.name}."
            )
        self._exterior_options[key] = value
        return self

    def add_safety_feature(self, feature: str) -> "CarBuilder":
        if feature not in self._spec.allowed_safety_features:
            raise InvalidConfigurationError(
                f"Safety feature '{feature}' is not offered on {self._spec.name}."
            )
        if feature not in self._safety_features:
            self._safety_features.append(feature)
        return self

    def build(self) -> Car:
        if self._spec.required_engine and self._engine is None:
            raise InvalidConfigurationError(f"{self._spec.name} requires an engine to be selected.")
        if self._spec.required_transmission and self._transmission is None:
            raise InvalidConfigurationError(f"{self._spec.name} requires a transmission to be selected.")
        missing_required_safety = self._spec.required_safety_features - set(self._safety_features)
        if missing_required_safety:
            raise InvalidConfigurationError(
                f"{self._spec.name} requires safety features {sorted(missing_required_safety)} "
                f"which have not been selected."
            )
        return Car(
            model=self._spec.name,
            engine=self._engine,
            transmission=self._transmission,
            interior_features=list(self._interior_features),
            exterior_options=dict(self._exterior_options),
            safety_features=list(self._safety_features),
        )


SEDAN_SPEC = CarModelSpec(
    name="Sedan LX",
    allowed_engines={"V6"},
    allowed_transmissions={"automatic", "manual"},
    allowed_interior_features={"leather seats", "GPS", "sound system"},
    allowed_exterior_options={"color", "rims"},
    allowed_safety_features={"ABS", "airbags", "rear camera"},
    required_safety_features={"ABS", "airbags"},
)

SUV_SPEC = CarModelSpec(
    name="SUV XT",
    allowed_engines={"V6", "V8"},
    allowed_transmissions={"automatic"},
    allowed_interior_features={"leather seats", "GPS", "sound system"},
    allowed_exterior_options={"color", "rims", "sunroof"},
    allowed_safety_features={"ABS", "airbags", "rear camera"},
    required_safety_features={"ABS", "airbags", "rear camera"},
)


# ---------------------------------------------------------------------------
# Part 3: Combined bonus — Car Management System facade
# ---------------------------------------------------------------------------

class CarManagementSystem:
    """Facade that turns a car configuration into an order document."""

    def __init__(self) -> None:
        self._editor = DocumentEditor()

    def configure_and_order(
        self,
        spec: CarModelSpec,
        configure_fn,
        export_format: str,
        save_path: str,
    ):
        builder = CarBuilder(spec)
        configure_fn(builder)
        car = builder.build()

        document = self._editor.new_document(export_format, car.summary())
        saved_path = self._editor.save(save_path)
        return car, document, saved_path


# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

def _configure_sedan(builder: CarBuilder) -> None:
    builder.set_engine("V6").set_transmission("automatic")
    builder.add_safety_feature("ABS").add_safety_feature("airbags")


def test_configure_and_order_pdf():
    system = CarManagementSystem()
    car, document, path = system.configure_and_order(
        spec=SEDAN_SPEC,
        configure_fn=_configure_sedan,
        export_format="pdf",
        save_path="/tmp/order",
    )
    assert car.model == "Sedan LX"
    assert isinstance(document, PDFDocument)
    assert path == "/tmp/order.pdf"
    assert "Sedan LX" in document.render()


def test_configure_and_order_word():
    system = CarManagementSystem()
    car, document, path = system.configure_and_order(
        spec=SEDAN_SPEC,
        configure_fn=_configure_sedan,
        export_format="word",
        save_path="/tmp/order",
    )
    assert isinstance(document, WordDocument)
    assert path.endswith(".docx")


def test_configure_and_order_html():
    def configure(builder: CarBuilder) -> None:
        builder.set_engine("V8").set_transmission("automatic")
        builder.add_safety_feature("ABS").add_safety_feature("airbags").add_safety_feature(
            "rear camera"
        )

    system = CarManagementSystem()
    car, document, path = system.configure_and_order(
        spec=SUV_SPEC,
        configure_fn=configure,
        export_format="html",
        save_path="/tmp/order",
    )
    assert isinstance(document, HTMLDocument)
    assert "SUV XT" in document.render()


def test_configure_and_order_propagates_invalid_configuration():
    def bad_configure(builder: CarBuilder) -> None:
        builder.set_engine("V6")  # transmission and safety features missing

    system = CarManagementSystem()
    with pytest.raises(InvalidConfigurationError):
        system.configure_and_order(
            spec=SEDAN_SPEC,
            configure_fn=bad_configure,
            export_format="pdf",
            save_path="/tmp/order",
        )


if __name__ == "__main__":
    # Two execution contexts are supported here:
    #
    # 1. Real script (`python test_car_management_system_standalone.py`):
    #    `__file__` exists and pytest can discover + run the tests in it
    #    normally via pytest.main([__file__]).
    #
    # 2. Pasted into a Colab/Jupyter cell:
    #    `__file__` does not exist, AND even if we skipped that, pytest
    #    discovers tests from files on disk -- it cannot see functions that
    #    only exist as in-memory objects in the notebook's namespace. So in
    #    this case we just call each `test_*` function directly and report
    #    pass/fail ourselves, with no dependency on pytest's file discovery.
    import sys

    try:
        _this_file = __file__
    except NameError:
        _this_file = None

    if _this_file:
        sys.exit(pytest.main([_this_file, "-v"]))
    else:
        test_functions = {
            name: obj
            for name, obj in list(globals().items())
            if name.startswith("test_") and callable(obj)
        }
        passed, failed = 0, []
        for name, fn in test_functions.items():
            try:
                fn()
                passed += 1
                print(f"PASSED  {name}")
            except Exception as exc:  # noqa: BLE001 - surfacing any failure
                failed.append((name, exc))
                print(f"FAILED  {name}  ->  {exc!r}")
        print(f"\n{passed} passed, {len(failed)} failed out of {len(test_functions)} tests")


PASSED  test_configure_and_order_pdf
PASSED  test_configure_and_order_word
PASSED  test_configure_and_order_html
PASSED  test_configure_and_order_propagates_invalid_configuration

4 passed, 0 failed out of 4 tests
